# Tune Thresholds

Chooses the two thresholds that `Segment_Specimens.ipynb` uses to tell beetles from look-alikes (bees, wasps...) and from things that aren't insects at all.

**The idea.** In `Segment_Specimens.ipynb`, Step 3b scores a random sample of boxes **once**: a *beetle score* and a *look-alike score* for each. Every pair of thresholds can then be tested on those saved scores instantly, with no model running. You supply the truth for a sample of boxes by sorting their pictures into three folders; the notebook counts the mistakes for every combination.

**The rule being tuned** (the same one as in Segment_Specimens):

| the box is... | when |
|---|---|
| a **look-alike** (bee, wasp...) → rejected | look-alike score ≥ `LOOKALIKE_MIN` **and** higher than the beetle score |
| a **beetle** → kept | otherwise, beetle score ≥ `BEETLE_MIN` |
| **not an insect** → rejected | otherwise (neither was confident) |

The lab's plan is a *low* `BEETLE_MIN` and a *high* `LOOKALIKE_MIN`, so real beetles are rarely lost.

**Steps**
1. Settings.
2. Load the scores.
3. Make a sample of crops to label, then **sort them by hand** into the `beetle`, `lookalike` and `other` folders (this is the one manual step).
4. Read your labels.
5. Try every threshold combination and pick the best.
6. Sort *all* scored crops into three folders for the best combinations, to look at the mistakes.
7. Check the rejection rate in each family (e.g. Staphylinidae).

**Kernel:** any environment with numpy, pandas, matplotlib and Pillow, such as the **specimen** one. No models are needed.

**Words used.** Beetle is the "positive" class. *True beetles rejected* are beetles wrongly thrown out (false negatives: lost training images). *Non-beetles passed* are bees, labels etc. wrongly kept (false positives: contamination). **Sensitivity** = share of true beetles kept. **Specificity** = share of non-beetles rejected.

## Step 1: Settings

- **`TUNING_DIR`**: the `tuning` folder that Step 3b made, inside `Data/<name>_segmented_extras/`. `None` opens a pop-up to choose it.
- **`LABEL_SAMPLE_SIZE`**: how many crops you will label by hand. More is better, especially for the rarer look-alikes. A few hundred is a reasonable start; **aim for at least 30 in each of `beetle` and `lookalike`**, or the estimates for that group are rough.
- **`BEETLE_MIN_VALUES`, `LOOKALIKE_MIN_VALUES`**: the thresholds to try. Every pair is tested.
- **`MIN_BEETLE_RECALL`**: for "Option B" below: keep at least this share of true beetles, then reject as many non-beetles as possible.
- **`SORT_COMBINATIONS`**: `(BEETLE_MIN, LOOKALIKE_MIN)` pairs to sort into folders in Step 6, e.g. `[(0.30, 0.85)]`. Leave empty to use the best two found.

In [ ]:
# ======================= EDIT THESE =======================

TUNING_DIR = None             # the "tuning" folder made by Step 3b; None -> pick with a pop-up

LABEL_SAMPLE_SIZE = 300       # crops to label by hand

BEETLE_MIN_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
LOOKALIKE_MIN_VALUES = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]

MIN_BEETLE_RECALL = 0.97      # Option B: keep at least 97% of true beetles
SORT_COMBINATIONS = []        # e.g. [(0.30, 0.85)]; empty -> the best two found

RANDOM_SEED = 42

# ==========================================================

## Step 2: Load the scores

Reads `candidate_scores.csv` (made by Step 3b) and defines the decision rule.

In [ ]:
import os
import shutil
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print


def find_repo_root(marker='Models'):
    path = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return os.path.abspath(os.getcwd())
        path = parent
    return os.path.abspath(os.getcwd())


def pick_folder(start_dir):
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    root.update()
    folder = filedialog.askdirectory(parent=root, title="Choose the 'tuning' folder made by Step 3b of Segment_Specimens",
                                     initialdir=start_dir)
    root.destroy()
    return folder


def decide_array(beetle, lookalike, beetle_min, lookalike_min):
    """The same rule as Segment_Specimens' decide(), for whole arrays of scores at once."""
    out = np.where(beetle >= beetle_min, "beetle", "no_insect")
    return np.where((lookalike >= lookalike_min) & (lookalike > beetle), "lookalike", out)


REPO_ROOT = find_repo_root()
DATA_DIR = os.path.join(REPO_ROOT, "Data")

if TUNING_DIR is None:
    print("A folder picker has opened. Check your taskbar if you don't see it.", flush=True)
    TUNING_DIR = pick_folder(DATA_DIR)
    if not TUNING_DIR:
        raise RuntimeError("No folder was chosen. Run this cell again and pick the tuning folder.")
if not os.path.isabs(TUNING_DIR):
    TUNING_DIR = os.path.join(DATA_DIR, TUNING_DIR)
TUNING_DIR = os.path.abspath(TUNING_DIR)

scores_path = os.path.join(TUNING_DIR, "candidate_scores.csv")
if not os.path.exists(scores_path):
    raise FileNotFoundError(f"No candidate_scores.csv in {TUNING_DIR}. Run Step 3b of Segment_Specimens.ipynb first.")

scores = pd.read_csv(scores_path, keep_default_na=False)
scores["crop"] = scores["id"].map(lambda i: os.path.join(TUNING_DIR, "candidates", i + ".jpg"))
scores["family"] = scores["source"].map(lambda s: os.path.basename(os.path.dirname(s)) or "(no subfolder)")

print(f"{len(scores)} scored boxes from {scores['source'].nunique()} photos.")
print(f"Beetle score:  median {scores['beetle'].median():.2f}   |   look-alike score: median {scores['lookalike'].median():.2f}")
print("Boxes per family:", dict(Counter(scores["family"]).most_common(8)), "...")

## Step 3: Make the labelling sample, then sort it by hand

This copies a random sample of crops into `tuning/labeling/unsorted/`. **Open that folder and move each picture into one of these folders** (next to `unsorted`):

- `beetle`: it is a beetle (any family)
- `lookalike`: a bee, wasp, ant, fly or other non-beetle insect
- `other`: not an insect (label, tape, scale bar, background, a fragment)

Crops that are too ambiguous to call can simply be left in `unsorted`; they are ignored. The file names don't show the model's scores, so you are labelling blind.

To label more later, raise `LABEL_SAMPLE_SIZE` and re-run this cell: it adds new crops and keeps your existing labels. Re-run it only when you want more; it does nothing if you already have enough.

In [ ]:
LABEL_DIR = os.path.join(TUNING_DIR, "labeling")
TRUTH_CLASSES = ["beetle", "lookalike", "other"]
for name in TRUTH_CLASSES + ["unsorted"]:
    os.makedirs(os.path.join(LABEL_DIR, name), exist_ok=True)

already = {os.path.splitext(f)[0] for name in TRUTH_CLASSES + ["unsorted"] for f in os.listdir(os.path.join(LABEL_DIR, name))}
want = min(LABEL_SAMPLE_SIZE, len(scores))
added = 0
if len(already) < want:
    pool = [i for i in scores["id"] if i not in already]
    random.Random(RANDOM_SEED).shuffle(pool)
    for cid in pool[:want - len(already)]:
        shutil.copy2(os.path.join(TUNING_DIR, "candidates", cid + ".jpg"), os.path.join(LABEL_DIR, "unsorted", cid + ".jpg"))
        added += 1

print(f"{added} crops added. Sort the pictures in:\n    {os.path.join(LABEL_DIR, 'unsorted')}")
print(f"into the folders beetle / lookalike / other next to it, then run the next cell.")

## Step 4: Read your labels

Reads which folder each crop is in and joins it to its scores.

In [ ]:
truth = {}
for name in TRUTH_CLASSES:
    for f in os.listdir(os.path.join(LABEL_DIR, name)):
        if f.lower().endswith(".jpg"):
            truth[os.path.splitext(f)[0]] = name
labels = pd.DataFrame({"id": list(truth), "truth": list(truth.values())})
labeled = scores.merge(labels, on="id")

counts = labeled["truth"].value_counts()
n_unsorted = len(os.listdir(os.path.join(LABEL_DIR, "unsorted")))
print(f"{len(labeled)} labelled crops: " + ", ".join(f"{counts.get(n, 0)} {n}" for n in TRUTH_CLASSES) + f"   ({n_unsorted} still unsorted)")

if counts.get("beetle", 0) == 0 or counts.get("lookalike", 0) + counts.get("other", 0) == 0:
    raise RuntimeError("Need at least one labelled beetle and one labelled non-beetle (lookalike or other).")
for name in ("beetle", "lookalike"):
    if counts.get(name, 0) < 30:
        print(f"WARNING: only {counts.get(name, 0)} labelled '{name}' crops; the numbers below will be rough for that group. Label more.")

# How well do the raw scores separate the labelled groups?
display(labeled.groupby("truth")[["beetle", "lookalike", "other"]].median().round(2))

## Step 5: Try every threshold combination

For every pair (`BEETLE_MIN`, `LOOKALIKE_MIN`), applies the rule to your labelled crops and counts:

- **true beetles rejected**: beetles that were thrown out (lost images)
- **non-beetles passed**: bees, labels etc. that were kept (contamination)
- **sensitivity** (share of true beetles kept), **specificity** (share of non-beetles rejected)
- **Youden's J** = sensitivity + specificity − 1. A perfect filter scores 1; one that is no better than guessing scores 0. This is the measure from the [Youden index page](https://metricgate.com/docs/youden-index/) that Tim suggested.

Two ways of picking the winner are shown:
- **Option A: best Youden's J.** Treats a lost beetle and a kept bee as equally bad.
- **Option B: highest specificity while keeping at least `MIN_BEETLE_RECALL` of true beetles.** Use this if losing real beetles is the bigger worry (as it is now). Check the counts: with a small sample, several combinations are practically tied, so prefer one in the middle of a good region of the maps over a lone best cell.

The table is also saved as `threshold_results.csv` for sharing.

In [ ]:
beetle_arr = labeled["beetle"].to_numpy()
look_arr = labeled["lookalike"].to_numpy()
is_beetle = (labeled["truth"] == "beetle").to_numpy()
n_beetle, n_other = int(is_beetle.sum()), int((~is_beetle).sum())

rows = []
for b in BEETLE_MIN_VALUES:
    for l in LOOKALIKE_MIN_VALUES:
        passed = decide_array(beetle_arr, look_arr, b, l) == "beetle"
        lost = int((~passed & is_beetle).sum())      # true beetles rejected
        leaked = int((passed & ~is_beetle).sum())    # non-beetles passed
        sens, spec = 1 - lost / n_beetle, 1 - leaked / n_other
        rows.append({"beetle_min": b, "lookalike_min": l,
                     "true_beetles_rejected": lost, "nonbeetles_passed": leaked,
                     "sensitivity": round(sens, 4), "specificity": round(spec, 4), "youden": round(sens + spec - 1, 4)})
results = pd.DataFrame(rows)
results.to_csv(os.path.join(TUNING_DIR, "threshold_results.csv"), index=False)

best_a = results.sort_values(["youden", "sensitivity"], ascending=False).iloc[0]
eligible = results[results["sensitivity"] >= MIN_BEETLE_RECALL]
best_b = eligible.sort_values(["specificity", "sensitivity"], ascending=False).iloc[0] if len(eligible) else None


def describe(r):
    return (f"BEETLE_MIN = {r.beetle_min:.2f}, LOOKALIKE_MIN = {r.lookalike_min:.2f}  ->  "
            f"{int(r.true_beetles_rejected)} of {n_beetle} true beetles rejected, "
            f"{int(r.nonbeetles_passed)} of {n_other} non-beetles passed  (sensitivity {r.sensitivity:.3f}, specificity {r.specificity:.3f}, Youden {r.youden:.3f})")


print("Option A (best Youden's J):")
print("   ", describe(best_a))
print(f"Option B (keep >= {MIN_BEETLE_RECALL:.0%} of true beetles, then best specificity):")
print("   ", describe(best_b) if best_b is not None else f"no combination keeps {MIN_BEETLE_RECALL:.0%} of true beetles; lower MIN_BEETLE_RECALL or label more.")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, col, title in zip(axes, ["sensitivity", "specificity", "youden"],
                          ["Sensitivity (share of true beetles kept)", "Specificity (share of non-beetles rejected)", "Youden's J"]):
    grid = results.pivot(index="beetle_min", columns="lookalike_min", values=col)
    im = ax.imshow(grid.to_numpy(), origin="lower", aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(grid.columns)))
    ax.set_xticklabels([f"{v:.2f}" for v in grid.columns], rotation=90, fontsize=7)
    ax.set_yticks(range(len(grid.index)))
    ax.set_yticklabels([f"{v:.2f}" for v in grid.index], fontsize=7)
    ax.set_xlabel("LOOKALIKE_MIN")
    ax.set_ylabel("BEETLE_MIN")
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046)
    for best, marker, label in ((best_a, "*", "A"), (best_b, "s", "B")):
        if best is not None:
            ax.plot(list(grid.columns).index(best.lookalike_min), list(grid.index).index(best.beetle_min),
                    marker, color="red", markersize=11, markerfacecolor="none", label=label)
axes[0].legend(title="best", fontsize=8)
plt.tight_layout()
plt.show()

display(results.sort_values(["youden", "sensitivity"], ascending=False).head(10).reset_index(drop=True))

## Step 6: Sort every scored crop into three folders

For each combination chosen, sorts **all** the scored crops (not just the ones you labelled) into

```
tuning/sorted/beetle0.30_lookalike0.85/
    1_no_insect_reject/          neither a confident beetle nor a look-alike
    2_lookalike_reject/<name>/   rejected as a bee, wasp, etc. (named by the best-matching look-alike)
    3_beetle_pass/               kept as beetles
    errors/true_beetles_rejected/   (labelled crops only) beetles that were thrown out, with where they went
    errors/nonbeetles_passed/       (labelled crops only) non-beetles that were kept, with what they really are
```

Flip through `errors/` first: it shows which kinds of beetle are being lost and which look-alikes get through. These folders are only for checking and can be deleted once the thresholds are settled; re-running this cell rebuilds them.

In [ ]:
combos = [(round(float(b), 2), round(float(l), 2)) for b, l in SORT_COMBINATIONS]
if not combos:
    combos = [(best_a.beetle_min, best_a.lookalike_min)]
    if best_b is not None:
        combos.append((best_b.beetle_min, best_b.lookalike_min))
combos = list(dict.fromkeys((round(float(b), 2), round(float(l), 2)) for b, l in combos))  # drop repeats

truth_by_id = dict(zip(labeled["id"], labeled["truth"]))


def sort_into_folders(b, l):
    base = os.path.join(TUNING_DIR, "sorted", f"beetle{b:.2f}_lookalike{l:.2f}")
    if os.path.isdir(base):
        shutil.rmtree(base)
    decisions = decide_array(scores["beetle"].to_numpy(), scores["lookalike"].to_numpy(), b, l)
    n = Counter()
    for row, d in zip(scores.itertuples(), decisions):
        if d == "beetle":
            folder = "3_beetle_pass"
        elif d == "lookalike":
            folder = os.path.join("2_lookalike_reject", row.lookalike_name)
        else:
            folder = "1_no_insect_reject"
        n[folder.split(os.sep)[0]] += 1
        os.makedirs(os.path.join(base, folder), exist_ok=True)
        shutil.copy2(row.crop, os.path.join(base, folder, row.id + ".jpg"))

        t = truth_by_id.get(row.id)
        error = None
        if t == "beetle" and d != "beetle":
            error = ("true_beetles_rejected", f"{row.id}__{d}.jpg")
        elif t is not None and t != "beetle" and d == "beetle":
            error = ("nonbeetles_passed", f"{row.id}__{t}.jpg")
        if error:
            n["errors/" + error[0]] += 1
            os.makedirs(os.path.join(base, "errors", error[0]), exist_ok=True)
            shutil.copy2(row.crop, os.path.join(base, "errors", error[0], error[1]))
    return base, n


for b, l in combos:
    base, n = sort_into_folders(b, l)
    print(f"BEETLE_MIN = {b:.2f}, LOOKALIKE_MIN = {l:.2f}  ->  {base}")
    for folder in sorted(n):
        print(f"    {folder}: {n[folder]}")

## Step 7: Rejection rate by family

For the first combination above, shows how many boxes in each family were kept or rejected, using **all** scored crops (labelled or not). Most boxes in a family should be beetles, so a family with a high rejection rate (Staphylinidae, for example) is a warning that real beetles are being lost, even if your labelled sample was too small to show it. Look at that family's crops in `errors/` or `1_no_insect_reject/` and `2_lookalike_reject/`.

In [ ]:
b, l = combos[0]
decision = pd.Series(decide_array(scores["beetle"].to_numpy(), scores["lookalike"].to_numpy(), b, l), index=scores.index)
table = pd.crosstab(scores["family"], decision).reindex(columns=["beetle", "lookalike", "no_insect"], fill_value=0)
table["boxes"] = table.sum(axis=1)
table["% rejected"] = (100 * (table["lookalike"] + table["no_insect"]) / table["boxes"]).round(1)
print(f"BEETLE_MIN = {b:.2f}, LOOKALIKE_MIN = {l:.2f}")
display(table.sort_values("% rejected", ascending=False))

print("\nWhen you have decided, set these in Step 1 of Segment_Specimens.ipynb:")
for b, l in combos:
    print(f"    BEETLE_MIN = {b:.2f}\n    LOOKALIKE_MIN = {l:.2f}\n")